In [ ]:
from collections.abc import Callable
from pathlib import Path

import equinox as eqx
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax
import xarray as xr

# from context_flux_no.models.fluxfno import FluxFNO1D
from context_flux_no.nn.operators.fourier_utils import append_grid_channels
from context_flux_no.training import PDEDataset, SegmentLoader
from einops import rearrange
from jaxtyping import Array, Float, PRNGKeyArray, PyTree


# jax.config.update("jax_enable_x64", True)

datadir = Path("../../data")

In [ ]:
dataset_xr = xr.load_dataset(datadir / "burgers_train.hdf5", engine="h5netcdf")
dataset_xr

In [ ]:
dataset = PDEDataset.from_xarray(dataset_xr.sel(t=slice(None, None, 10)))

In [ ]:
test_arr = jnp.reshape(jnp.arange(10.0), (1, -1))
print(test_arr.shape)
stencil_size = (2, 2)
test_padded = jnp.pad(test_arr, pad_width=[(0, 0), stencil_size], mode="wrap")
print(test_padded.shape)

In [ ]:
jax.lax.conv_general_dilated_patches(
    jnp.expand_dims(test_padded, axis=0), (sum(stencil_size) + 1,), (1,), "valid"
).shape

In [ ]:
class FluxNO1D(eqx.Module):
    in_channels: int = eqx.field(static=True)
    stencil_widths: tuple[int, int] = eqx.field(static=True)
    lift_dim: int = eqx.field(static=True)
    hidden_dim: int = eqx.field(static=True)
    depth: int = eqx.field(static=True)

    lift_layer: eqx.nn.Conv1d
    mlp: eqx.nn.MLP

    def __init__(
        self,
        in_channels: int,
        stencil_widths: tuple[int, int],
        lift_dim: int,
        hidden_dim: int,
        depth: int,
        dtype=None,
        *,
        key: PRNGKeyArray,
    ):
        keys = jax.random.split(key, 2)
        kernel_size = stencil_widths[0] + stencil_widths[1] + 1
        self.lift_layer = eqx.nn.Conv1d(
            in_channels=in_channels + 1,
            out_channels=lift_dim,
            kernel_size=kernel_size,
            dtype=dtype,
            key=keys[0],
        )
        self.mlp = eqx.nn.MLP(
            in_size=lift_dim,
            out_size=in_channels,
            width_size=hidden_dim,
            depth=depth,
            activation=jax.nn.gelu,
            dtype=dtype,
            key=keys[1],
        )
        self.in_channels = in_channels
        self.stencil_widths = stencil_widths
        self.lift_dim = lift_dim
        self.hidden_dim = hidden_dim
        self.depth = depth

    @property
    def stencil_size(self) -> int:
        return sum(self.stencil_widths) + 1

    def get_stencils(
        self, u: Float[Array, "channels grid_x"]
    ) -> Float[Array, "lift_dim grids_x+1"]:
        a, b = self.stencil_widths
        pad_widths = [(0, 0), (a + 1, b)]
        # Need to change mode if not periodic boundary condition
        u_padded = jnp.pad(u, pad_widths, mode="wrap")
        u_stencils = jax.lax.conv_general_dilated_patches(
            jnp.expand_dims(u_padded, axis=0),  # Requires a dummy batch axis
            filter_shape=(self.stencil_size,),
            window_strides=(1,),
            padding="valid",
        )[0]  # Remove the batch axis
        return rearrange(u_stencils, "(c s) ... -> c s ...", c=u.shape[0])

    def compute_flux(
        self, u_stencils: Float[Array, "channels {self.stencil_size} grids_x+1"]
    ) -> Float[Array, "channels grids_x+1"]:
        pass

    def __call__(
        self,
        u: Float[Array, "channels grids_x"],
        dt: float,
        dx: float,
        *,
        key: PRNGKeyArray | None = None,
    ) -> Float[Array, "channels grids_x"]:
        a, b = self.stencil_widths
        pad_widths = [(0, 0), (a + 1, b)]
        # Need to change mode if not periodic boundary condition
        u_padded = jnp.pad(append_grid_channels(u), pad_widths, mode="wrap")
        u_stencils: Float[Array, "lift_dim grids_x+1"] = self.lift_layer(u_padded)
        f: Float[Array, "channels grids_x+1"] = eqx.filter_vmap(
            self.mlp, in_axes=-1, out_axes=-1
        )(u_stencils)
        du_dt: Float[Array, "channels grids_x"] = -jnp.diff(f, axis=1) / dx
        return u + du_dt * dt

In [ ]:
def train(
    model: eqx.Module,
    dataloader: SegmentLoader,
    optimizer: optax.GradientTransformation,
    loss_fn: Callable[[eqx.Module, PyTree[Array]], Float[Array, ""]],
    max_steps: int,
):
    loss_grad_fn = eqx.filter_value_and_grad(loss_fn, has_aux=True)

    @eqx.filter_jit
    def train_step(model_, loader_state_, opt_state_):
        batch, loader_state_ = dataloader.load_batch(loader_state_)

        (loss, aux), grads = loss_grad_fn(model_, batch)
        updates, opt_state_ = optimizer.update(
            grads, opt_state_, eqx.filter(model_, eqx.is_array)
        )
        model_ = eqx.apply_updates(model_, updates)
        return loss, aux, model_, loader_state_, opt_state_

    loader_state = dataloader.init()
    opt_state = optimizer.init(eqx.filter(model, eqx.is_array))
    loss_history = []
    for i in range(max_steps):
        loss, aux, model, loader_state, opt_state = train_step(
            model, loader_state, opt_state
        )
        ## Logging and save logic
        loss_ = loss.item()
        print(f"Step: {i}: loss = {loss_}")
        loss_history.append(loss_)
    return model, jnp.asarray(loss_history)

In [ ]:
loader = SegmentLoader(dataset, 2, 1024)

In [ ]:
loader_state = loader.init()
batch, loader_state = loader.load_batch(loader_state)

In [ ]:
batch

In [ ]:
model = FluxNO1D(
    in_channels=1,
    lift_dim=64,
    hidden_dim=64,
    depth=4,
    stencil_widths=(8, 8),
    key=jax.random.key(0),
)

In [ ]:
from typing import Any


def mean_squared_error(model, batch) -> tuple[Float[Array, ""], Any]:
    u_batch, dt, dx = batch
    u0, u1 = u_batch[:, 0], u_batch[:, 1]
    u1_pred: Float[Array, "batch dim x"] = eqx.filter_vmap(
        model, in_axes=(0, None, None)
    )(u0, dt, dx)
    return jnp.mean((u1_pred - u1) ** 2), None

In [ ]:
u_batch[0, 0].shape

In [ ]:
u_batch, dt, dx = batch
model(u_batch[0, 0], dt, dx).shape

In [ ]:
u_batch[:, 0].shape

In [ ]:
dx

In [ ]:
mean_squared_error(model, batch)

In [ ]:
model_trained, loss_history = train(
    model, loader, optax.adabelief(1e-3), mean_squared_error, max_steps=2000
)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 5))
ax.plot(loss_history)
ax.set_yscale("log", base=10)
ax.set_xlabel("Train steps")
ax.set_ylabel("Train loss")

In [ ]:
sample_idx = 0
time_idx = 500
u0 = batch[0][sample_idx, 0]
u1 = batch[0][sample_idx, 1]
dt = batch[1]

dx = batch[2]


fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True, height_ratios=(4, 1))

u1_pred = model_trained(u0, dt, dx)[0]
axes[0].plot(jnp.arange(len(u1[0])) * dx, u1[0], "--", label="Data")
axes[0].plot(jnp.arange(len(u1[0])) * dx, u1_pred, label="Flux FNO")
axes[0].legend()

axes[0].set_ylabel("$u$")
axes[0].set_title(f"State prediction: $t$ = {time_idx * float(dt):3.3f}")
axes[1].plot(jnp.arange(len(u1[0])) * dx, u1_pred - u1[0])
axes[1].set_xlabel("$x$")
axes[1].set_ylabel("$u_{pred}-u_{data}$")

In [ ]:
time_idx = 100
f_pred = model_trained.physical_flux(u0)
f_true = 0.5 * u0**2
fig, ax = plt.subplots(1, 1, figsize=(7, 5))
ax.plot(x, f_true[0], label="True flux")
ax.plot(x, f_pred[0], label="Learned flux")
ax.legend()
ax.set_xlabel("$x$")
ax.set_title(f"Flux prediction: $t$ = {time_idx * float(dt):3.3f}")